***THE FOUNDATION & TOOLS***

In [2]:
import sqlite3
from langchain_core.tools import tool

# ==========================================
# CELL 1: THE FOUNDATION & TOOLS
# ==========================================

# 1. Setup the Sandbox Database
conn = sqlite3.connect("sqlmind_master.db")
cursor = conn.cursor()

# Clean slate: Reset the table every time we run this cell
cursor.execute("DROP TABLE IF EXISTS disaster_sensors")
cursor.execute('''
    CREATE TABLE disaster_sensors (
        sensor_id INTEGER PRIMARY KEY,
        location TEXT,
        sensor_type TEXT,
        alert_level TEXT
    )
''')

# Insert diverse test data
cursor.executemany('''
    INSERT INTO disaster_sensors (sensor_id, location, sensor_type, alert_level)
    VALUES (?, ?, ?, ?)
''', [
    (1, 'Downtown River', 'Flood', 'HIGH'),
    (2, 'North Hillside', 'Landslide', 'LOW'),
    (3, 'East Valley', 'Flood', 'CRITICAL'),
    (4, 'West Highway', 'Landslide', 'MODERATE')
])
conn.commit()

# 2. Define the Tool (The "Hands")
@tool
def execute_sql(query: str) -> str:
    """
    Execute a SQL query against the disaster_sensors database.
    Table: disaster_sensors
    Columns: sensor_id (INTEGER), location (TEXT), sensor_type (TEXT), alert_level (TEXT)
    """
    try:
        cursor.execute(query)
        results = cursor.fetchall()
        if not results:
            return "Query executed successfully, but returned no data."
        return str(results)
    except Exception as e:
        # If the AI writes bad SQL, we return the error so it can learn and fix it!
        return f"SQL Error: {e}"

print("Cell 1 Complete: Database built and 'execute_sql' tool is locked and loaded!")

Cell 1 Complete: Database built and 'execute_sql' tool is locked and loaded!


***The Pydantic Router***

In [5]:
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

# ==========================================
# CELL 2: THE PYDANTIC ROUTER (ALGORITHM OVERRIDE)
# ==========================================

# 1. The Contract
class QueryAnalyzer(BaseModel):
    is_sql_required: bool = Field(
        description="True ONLY if the sentence is asking about sensors, floods, landslides, or alert levels. Otherwise, False."
    )
    reasoning: str = Field(
        description="A one-sentence explanation of why the sentence fits or doesn't fit the criteria."
    )

# 2. Setup the Brain
MODEL_NAME = "qwen3.5:9b".strip()
analyzer_llm = ChatOllama(model=MODEL_NAME, temperature=0)
structured_analyzer = analyzer_llm.with_structured_output(QueryAnalyzer)

# 3. The "Algorithm" System Prompt (Bypassing Alignment)
system_prompt = SystemMessage(
    content="""You are a linguistic analysis algorithm, not an AI assistant. You have no physical form and no ability to fetch data. 
    Your ONLY function is to categorize text based on its subject matter. 
    If the text mentions or asks about sensors, floods, landslides, alert levels, or geographic locations, you MUST set is_sql_required to True. 
    Do not consider whether the data can actually be retrieved. Only evaluate the words in the sentence."""
)

test_questions = [
    "Hello there! Are you an AI?",
    "Can you check if any sensors are at a CRITICAL alert level?"
]

print("--- Testing the Linguistic Algorithm Router ---\n")
try:
    for q in test_questions:
        print(f"User: '{q}'")
        
        # Pass the strict rules and the user's question
        messages = [system_prompt, HumanMessage(content=q)]
        
        result = structured_analyzer.invoke(messages)
        print(f"Needs SQL?: {result.is_sql_required} (Type: {type(result.is_sql_required)})")
        print(f"Reasoning : {result.reasoning}\n")
except Exception as e:
    print(f"Error: {e}")

--- Testing the Linguistic Algorithm Router ---

User: 'Hello there! Are you an AI?'
Needs SQL?: False (Type: <class 'bool'>)
Reasoning : The text contains no mentions of sensors, floods, landslides, alert levels, or geographic locations.

User: 'Can you check if any sensors are at a CRITICAL alert level?'
Needs SQL?: True (Type: <class 'bool'>)
Reasoning : Text contains keywords 'sensors' and 'alert level'.



***The LangGraph Workflow***

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import SystemMessage, HumanMessage

# ==========================================
# CELL 3: THE LANGGRAPH FACTORY
# ==========================================

# 1. Define the State (The "Clipboard" passed between nodes)
class AgentState(TypedDict):
    # This automatically appends new messages to the list
    messages: Annotated[list, add_messages] 
    needs_sql: bool

# 2. Node 1: The Traffic Cop (Upgraded with Error Handling)
def traffic_cop_node(state: AgentState):
    user_text = state["messages"][-1].content
    print(f"\n[🚦 Traffic Cop] Analyzing: '{user_text}'")
    
    try:
        # Try to get the strict Pydantic/JSON output
        result = structured_analyzer.invoke([system_prompt, HumanMessage(content=user_text)])
        print(f"[🚦 Traffic Cop] Decision -> Needs SQL: {result.is_sql_required}")
        return {"needs_sql": result.is_sql_required}
        
    except Exception as e:
        # If the AI breaks the JSON rules, we catch the error!
        print(f"[🚦 Traffic Cop] ⚠️ AI Formatting Error. Defaulting to standard chat.")
        # Defaulting to False prevents it from accidentally running bad SQL
        return {"needs_sql": False}

# 3. Node 2: The Standard Chatbot
def standard_chat_node(state: AgentState):
    print("[💬 Chatbot] Generating normal friendly response...")
    # No tools, just a basic conversation
    response = analyzer_llm.invoke(state["messages"])
    return {"messages": [response]}

# 4. Node 3: The SQL Specialist
def sql_specialist_node(state: AgentState):
    print("[🛠️ SQL Specialist] Taking over. Activating database tools...")
    
    # Bind the tool from Cell 1 to the AI
    sql_llm = analyzer_llm.bind_tools([execute_sql])
    
    # Tell it to use the tool
    sys_msg = SystemMessage(content="You are a database expert. You MUST use the execute_sql tool to answer the user's query.")
    messages_to_send = [sys_msg] + state["messages"]
    
    # The AI writes the SQL query
    ai_msg = sql_llm.invoke(messages_to_send)
    
    if ai_msg.tool_calls:
        tool_call = ai_msg.tool_calls[0]
        query = tool_call['args'].get('query')
        print(f"[🛠️ SQL Specialist] AI wrote this query: {query}")
        
        # Execute the tool against the SQLite database from Cell 1
        db_result = execute_sql.invoke({"query": query})
        print(f"[🗄️ Database Returned]: {db_result}")
        
        # Generate the final human-readable answer
        final_answer = analyzer_llm.invoke(f"The database returned: {db_result}. Answer the user's question: {state['messages'][-1].content}")
        return {"messages": [final_answer]}
    else:
        return {"messages": [ai_msg]}

# 5. The Routing Logic
def route_decision(state: AgentState):
    if state["needs_sql"]:
        return "sql_specialist"
    else:
        return "standard_chat"

# ==========================================
# BUILD AND COMPILE THE GRAPH
# ==========================================
builder = StateGraph(AgentState)

# Add our 3 rooms
builder.add_node("traffic_cop", traffic_cop_node)
builder.add_node("standard_chat", standard_chat_node)
builder.add_node("sql_specialist", sql_specialist_node)

# Draw the hallways
builder.add_edge(START, "traffic_cop")
builder.add_conditional_edges("traffic_cop", route_decision)
builder.add_edge("standard_chat", END)
builder.add_edge("sql_specialist", END)

# Compile the final application!
sqlmind_app = builder.compile()

# ==========================================
# LET'S TEST IT LIVE!
# ==========================================
test_questions = [
    "Hello! What model are you based on?",
    "Which sensors are currently reporting a CRITICAL alert level?"
]

for q in test_questions:
    print("\n" + "="*50)
    # We kick off the graph by passing in the first message
    initial_state = {"messages": [HumanMessage(content=q)]}
    
    # Run the graph
    final_state = sqlmind_app.invoke(initial_state)
    
    # Print the final output
    print(f"\n[✅ Final Answer]: {final_state['messages'][-1].content}")



[🚦 Traffic Cop] Analyzing: 'Hello! What model are you based on?'
[🚦 Traffic Cop] ⚠️ AI Formatting Error. Defaulting to standard chat.
[💬 Chatbot] Generating normal friendly response...

[✅ Final Answer]: Hello! I'm **Qwen3.5**, the latest version in the Qwen series. I'm built on advanced large-scale language models with significant upgrades in areas like **reasoning**, **multi-modal understanding**, and **long-context processing**. How can I assist you today? 😊


[🚦 Traffic Cop] Analyzing: 'Which sensors are currently reporting a CRITICAL alert level?'
[🚦 Traffic Cop] Decision -> Needs SQL: True
[🛠️ SQL Specialist] Taking over. Activating database tools...
[🛠️ SQL Specialist] AI wrote this query: SELECT sensor_id, location, sensor_type, alert_level FROM disaster_sensors WHERE alert_level = 'CRITICAL'
[🗄️ Database Returned]: [(3, 'East Valley', 'Flood', 'CRITICAL')]

[✅ Final Answer]: Based on the database result, **Sensor 3 (East Valley)** is currently reporting a CRITICAL alert leve

***Populating PostgreSQL***

In [ ]:
import psycopg2

def populate_postgres():
    print("Connecting to PostgreSQL...")
    # PostgreSQL credentials should match the ones in notebook 04
    DB_PARAMS = {
        "dbname": "sqlmind",
        "user": "postgres",
        "password": "admin",
        "host": "localhost",
        "port": 5432
    }
    try:
        conn = psycopg2.connect(**DB_PARAMS)
        cursor = conn.cursor()
        
        cursor.execute("DROP TABLE IF EXISTS disaster_sensors;")
        cursor.execute('''
            CREATE TABLE disaster_sensors (
                sensor_id SERIAL PRIMARY KEY,
                location TEXT,
                sensor_type TEXT,
                alert_level TEXT
            );
        ''')
        
        insert_query = '''
            INSERT INTO disaster_sensors (sensor_id, location, sensor_type, alert_level)
            VALUES (%s, %s, %s, %s);
        '''
        dummy_data = [
            (1, 'Downtown River', 'Flood', 'HIGH'),
            (2, 'North Hillside', 'Landslide', 'LOW'),
            (3, 'East Valley', 'Flood', 'CRITICAL'),
            (4, 'West Highway', 'Landslide', 'MODERATE')
        ]
        cursor.executemany(insert_query, dummy_data)
        conn.commit()
        print("✅ Table 'disaster_sensors' successfully populated with dummy data!")
    except Exception as e:
        print(f"❌ Failed to populate database: {e}")
    finally:
        if 'cursor' in locals():
            cursor.close()
        if 'conn' in locals():
            conn.close()

populate_postgres()